# Volume 6, Chapter 3: Topological Spaces (Python companion)

Exercises first, solutions at the end. Topology on a computer: counting the topologies on a few points, connected components by union–find and the percolation threshold, the connectivity paradox of digital images, Furstenberg's metric on the integers, and Tietze's extension built from Urysohn functions. Requirements: numpy, scipy, sympy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P3.1 (★) Counting topologies.** By brute force, list all topologies on a set with $n=1,2,3,4$ points and count those that are $T_0$. Then count preorders on $n\le5$ points and compare (Exercise 3.6). How many topologies on $3$ points are there up to homeomorphism?

**P3.2 (★★) CS application: connected components and percolation.** Open each site of an $L\times L$ grid independently with probability $p$, and join neighbouring open sites. Compute the clusters with a *union–find* structure (disjoint sets with path compression and union by size). Estimate the probability that an open cluster crosses from the top row to the bottom row, as a function of $p$, for $L=20,50,100$. Where does the curve jump?

**P3.3 (★★) Digital topology.** In a binary image, the black pixels can be called connected through their $4$ edge-neighbours or through their $8$ edge-or-corner neighbours. Count the black and white components of a digital circle and of a diagonal line with each convention, and find the configuration where the digital "Jordan curve theorem" fails unless black and white use *different* conventions.

**P3.4 (★) Furstenberg's metric.** Implement $v(n)$ and $d(m,n)=2^{-v(m-n)}$ of Exercise 3.5. Check the ultrametric inequality on random triples, and that the ball $B(a,2^{-k})$ restricted to $[-1000,1000]$ is $a+(k+1)!\,\Z$. Then list the integers of $[-1000,1000]$ outside $\bigcup_{p\le P}p\Z$ for $P=2,3,5,7,11$: what would they be if there were no prime beyond $P$?

**P3.5 (★★) Tietze's extension, step by step.** Let $A=[-2,-1]\cup[1,2]\cup\{0\}$ in $\R$ and $f=\sin(3x)$ on $[-2,-1]$, $f=x-\frac32$ on $[1,2]$, $f(0)=1$. Run the construction of Exercise 3.8, with Urysohn's function $u=\frac{d(x,B)}{d(x,B)+d(x,C)}$, for $20$ steps on a grid of $[-3,3]$. Plot $F$ and check the error on $A$ against $(\frac23)^n$.

## Solutions

### P3.1

In [ ]:
from itertools import combinations, product, permutations

def topologies(n):
    pts = range(n)
    subsets = [frozenset(c) for r in range(n + 1) for c in combinations(pts, r)]
    full, empty = frozenset(pts), frozenset()
    middle = [s for s in subsets if s not in (full, empty)]
    found = []
    for mask in range(2 ** len(middle)):
        T = {empty, full} | {middle[i] for i in range(len(middle)) if mask >> i & 1}
        if all(a | b in T and a & b in T for a in T for b in T):
            found.append(frozenset(T))
    return found

def is_T0(T, n):
    return all(any((x in U) != (y in U) for U in T) for x in range(n) for y in range(x + 1, n))

def preorders(n):
    pairs = [(x, y) for x in range(n) for y in range(n) if x != y]
    count = 0
    for bits in product([0, 1], repeat=len(pairs)):
        R = {p for p, b in zip(pairs, bits) if b}
        if all((x, z) in R for (x, y) in R for (y2, z) in R if y == y2 and x != z):
            count += 1
    return count

for n in range(1, 5):
    T = topologies(n)
    print(n, "points:", len(T), "topologies,", sum(is_T0(t, n) for t in T), "of them T0")
print("preorders on 1..5 points:", [preorders(n) for n in range(1, 6)])

T3 = topologies(3)
classes = {min(tuple(sorted(tuple(sorted(sigma[x] for x in U)) for U in t)) for sigma in permutations(range(3))) for t in T3}
print("topologies on 3 points up to homeomorphism:", len(classes))

The counts are $1,4,29,355$ topologies and $1,3,19,219$ $T_0$ topologies, and the preorders are $1,4,29,355,6942$: the bijection of Exercise 3.6 in numbers. $T_0$ topologies are the *partial orders* (antisymmetric preorders): $1,3,19,219$ is the number of posets on labelled points. Up to homeomorphism, $3$ points carry $9$ topologies. No formula is known for the number of topologies on $n$ points (OEIS A000798); brute force over $2^{2^n-2}$ families dies at $n=5$, where $2^{30}$ families must be tested, but the preorder count still works because it only enumerates relations.

### P3.2

In [ ]:
import numpy as np

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n)); self.size = [1] * n
    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:          # path compression
            self.parent[x], x = root, self.parent[x]
        return root
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b: return
        if self.size[a] < self.size[b]: a, b = b, a
        self.parent[b] = a; self.size[a] += self.size[b]

def crosses(L, p, rng):
    open_ = rng.random((L, L)) < p
    TOP, BOTTOM = L * L, L * L + 1               # two virtual sites
    uf = UnionFind(L * L + 2)
    for i in range(L):
        for j in range(L):
            if not open_[i, j]: continue
            k = i * L + j
            if i == 0: uf.union(k, TOP)
            if i == L - 1: uf.union(k, BOTTOM)
            if i + 1 < L and open_[i + 1, j]: uf.union(k, k + L)
            if j + 1 < L and open_[i, j + 1]: uf.union(k, k + 1)
    return uf.find(TOP) == uf.find(BOTTOM)

rng = np.random.default_rng(0)
ps = np.round(np.arange(0.45, 0.76, 0.03), 2)
for L, trials in [(20, 400), (50, 200), (100, 80)]:
    probs = [np.mean([crosses(L, p, rng) for _ in range(trials)]) for p in ps]
    print(f"L={L:3d}:", " ".join(f"{p}:{q:.2f}" for p, q in zip(ps, probs)))

The crossing probability climbs from almost $0$ to almost $1$ around $p\approx0.59$, and the climb gets steeper as $L$ grows: in the limit it becomes a jump at the critical value $p_c\approx0.5927$ of site percolation on the square lattice, which is known only numerically. (For *bond* percolation on the same lattice Kesten proved $p_c=\frac12$ exactly in 1980.) Union–find runs in time $O(L^2\,\alpha(L^2))$ where $\alpha$ is the inverse Ackermann function, at most $4$ for any grid that fits in a computer (Tarjan, 1975). It is the algorithm behind connected-component labelling of images and Kruskal's minimum spanning tree: connectedness, made computational.

### P3.3

In [ ]:
import numpy as np
from scipy import ndimage

N4 = ndimage.generate_binary_structure(2, 1)   # edges only
N8 = ndimage.generate_binary_structure(2, 2)   # edges and corners

def count(img):
    out = {}
    for name, s in [("4", N4), ("8", N8)]:
        out["black-" + name] = ndimage.label(img, structure=s)[1]
        out["white-" + name] = ndimage.label(~img, structure=s)[1]
    return out

n = 21
yy, xx = np.mgrid[:n, :n]
r = np.hypot(xx - 10, yy - 10)
circle = np.abs(r - 7) < 0.5                   # a thin digital circle
diamond = (np.abs(xx - 10) + np.abs(yy - 10)) == 6   # a "circle" made of diagonal steps
print("thin circle:", count(circle))
print("diamond:    ", count(diamond))
for row in diamond[2:19, 2:19].astype(int):
    print("".join(".#"[v] for v in row))

The diamond is a closed curve of diagonal steps. With $8$-connectivity for both colours it is one black component, yet the white inside and outside also meet through the corners: one white component, so the curve does not separate the plane. With $4$-connectivity for both, the black curve falls apart into $24$ isolated pixels, yet the white still splits into two components. Neither convention alone gives the Jordan curve theorem. Using $8$ for black and $4$ for white (or the reverse) restores it (Rosenfeld, 1970s). A cleaner fix is the *Khalimsky topology*, a genuine (non-Hausdorff) topology on $\Z^2$ built from the finite spaces of P3.1, in which a digital Jordan theorem is a theorem.

### P3.4

In [ ]:
import math, random
from sympy import primerange

def v(n):
    if n == 0: return math.inf
    k = 1
    while n % math.factorial(k + 1) == 0:
        k += 1
    return k

def d(m, n):
    return 0.0 if m == n else 2.0 ** -v(m - n)

random.seed(1)
bad = 0
for _ in range(20000):
    x, y, z = (random.randint(-10**6, 10**6) for _ in range(3))
    bad += d(x, z) > max(d(x, y), d(y, z))
print("ultrametric violations:", bad)

R = range(-1000, 1001)
for a, k in [(7, 2), (-5, 3), (11, 4)]:
    ball = {n for n in R if d(a, n) < 2.0 ** -k}
    prog = {n for n in R if (n - a) % math.factorial(k + 1) == 0}
    print(f"B({a}, 2^-{k}) == {a} + {k+1}!Z on [-1000,1000]:", ball == prog)

for P in [2, 3, 5, 7, 11]:
    ps = list(primerange(2, P + 1))
    left = [n for n in R if all(n % p for p in ps)]
    print(f"P={P:2d}: {len(left)} integers left, smallest positive ones:", [n for n in left if n > 0][:8])

No violation of the ultrametric inequality, and the balls are the progressions $a+(k+1)!\Z$, as in the solution of Exercise 3.5. If $2,3,5,7,11$ were all the primes, the integers outside $\bigcup p\Z$ would be exactly $\pm1$ (every other integer has a prime factor). The computer finds $13,17,19,\ldots$ instead: the complement of a finite union of closed-open progressions is open, so it is infinite, while $\{\pm1\}$ is finite. Furstenberg's argument says that the leftover set can never shrink to $\{\pm1\}$; the computation shows what it contains instead, namely the numbers whose prime factors are all larger than $P$.

### P3.5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-3, 3, 6001)
onA = (np.abs(np.abs(x) - 1.5) <= 0.5 + 1e-12) | (np.abs(x) < 1e-12)
f = np.where(x < -0.5, np.sin(3 * x), np.where(x > 0.5, x - 1.5, 1.0))

def dist_to(mask):
    pts = x[mask]
    if len(pts) == 0: return None
    return np.min(np.abs(x[:, None] - pts[None, :]), axis=1)

def step(h, s):
    """One Urysohn correction for h on A with |h| <= s: returns g with |g| <= s/3 and |h-g| <= 2s/3 on A."""
    dB, dC = dist_to(onA & (h <= -s / 3)), dist_to(onA & (h >= s / 3))
    if dB is None and dC is None: return np.zeros_like(x)
    if dB is None: return np.full_like(x, s / 3)
    if dC is None: return np.full_like(x, -s / 3)
    u = dB / (dB + dC)
    return s * (2 * u / 3 - 1 / 3)

F = np.zeros_like(x); s = 1.0
for n in range(1, 21):
    F += step(f - F, s); s *= 2 / 3
    if n in (1, 2, 5, 10, 20):
        print(f"n={n:2d}: max error on A = {np.max(np.abs(f - F)[onA]):.2e}   bound (2/3)^n = {s:.2e}")

plt.figure(figsize=(7, 3))
plt.plot(x, F, label="extension F")
plt.plot(x[onA], f[onA], ".", ms=1.5, label="f on A")
plt.legend(); plt.title("Tietze extension after 20 Urysohn corrections"); plt.show()

The error on $A$ is never above $(\frac23)^n$, as Exercise 3.8(b) proves, and here it is *equal* to it at every step: the bound is sharp. The culprit is the isolated point $0$, where $f=1$: each correction can move $F(0)$ only by $\frac13$ of the current size $s$, and the residual at $0$ is always exactly $s$. Outside $A$ every correction is built from distance functions, so it is continuous, and the uniform limit is continuous; it interpolates across the gaps $]-3,-2[$, $]-1,0[$, $]0,1[$, $]2,3[$ and never leaves $[-1,1]$. Tietze's theorem says *some* extension exists; numerical analysts prefer the explicit McShane formula $\inf_{a\in A}(f(a)+L|x-a|)$ for $L$-Lipschitz data, which preserves the Lipschitz constant.